# Promotion comparisons and post-promotion patterns

Reference: **Two_Analytics_Projects_Complete_Guide.docx, Steps 1.4.1–1.4.4**. SQL: `sql/04_promo.sql`; runner: `scripts/run_promotions.py`. The notebook reads exports without locking DBeaver.

These are associations in panel purchases, not causal effects or total store sales.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / 'exports/promotions/promo_lift.csv').exists():
    ROOT = ROOT / 'outputs/grocery-marketing-effectiveness'
DATA = ROOT / 'exports/promotions'
coverage = pd.read_csv(DATA / 'promotion_coverage.csv')
lift = pd.read_csv(DATA / 'promo_lift.csv')
post = pd.read_csv(DATA / 'post_promo_lift.csv')
positive = pd.read_csv(DATA / 'promo_lift_positive_baseline.csv')
observed = pd.read_csv(DATA / 'promo_lift_observed_only.csv')
checks = pd.read_csv(DATA / 'promotion_checks.csv')
assert checks.passed.astype(str).str.lower().eq('true').all()
assert int(coverage.eligible_pairs.iloc[0]) == 22776
assert int(coverage.pair_weeks.iloc[0]) == 1184352
assert int(lift.pair_weeks.sum()) == 52 * int(coverage.baseline_pairs.iloc[0])
print(f'{len(checks)} full-data checks passed')
display(coverage.T)

## 1. Product/store/week panel

Pairs must have purchases in at least 8 weeks. Each receives source weeks 2–53. Missing purchase records mean zero panel purchases, not proven zero store sales. Baselines require at least 4 weeks without recorded display/mailer and not immediately after a promotion.

In [ ]:
display(pd.read_csv(DATA / 'panel_sample.csv', dtype={'product_id': str, 'store_id': str}).head(15))
display(lift)
display(Image(filename=str(ROOT / 'charts/promotion_lift.png')))

## 2. Placement comparisons

Display-only and mailer-only groups are analysed separately. MULTIPLE preserves conflicting source codes. Counts below 100 are flagged and suppressed only in the chart; full exports retain them.

In [ ]:
display(pd.read_csv(DATA / 'display_lift.csv', dtype={'placement_code': str}))
display(pd.read_csv(DATA / 'mailer_lift.csv', dtype={'placement_code': str}))
display(Image(filename=str(ROOT / 'charts/promotion_placements.png')))

## 3. Weeks after promotions

A single-week sequence means the previous week was promoted but the week before it was not recorded as promoted. Consecutive-week sequences have both previous weeks flagged. Boundary history is incomplete and counted explicitly. These comparisons do not isolate stockpiling or prove that there is no pull-forward effect.

In [ ]:
display(post)
display(Image(filename=str(ROOT / 'charts/post_promotion.png')))

## 4. Baseline and coverage sensitivity

The guide treats absent promotion rows as no recorded display/mailer. Every explicitly observed eligible record is promotional; therefore the observed-metadata-only baseline is unsupported, not a zero effect. Positive-baseline sensitivity is estimable and reported separately. Do not average indices across incompatible populations.

In [ ]:
assert observed.empty
assert int(coverage.observed_only_baseline_pairs.iloc[0]) == 0
print('Observed-metadata-only comparison: NOT ESTIMABLE — no explicitly observed nonpromotion baselines.')
comparison = lift[['condition','lift_index']].merge(positive[['condition','lift_index']], on='condition', suffixes=('_primary','_positive_baseline'))
display(comparison)
display(checks)

## 5. Findings and next step

In [ ]:
display(Markdown((ROOT / 'docs/promotions/findings.md').read_text(encoding='utf-8')))

Next in the Word guide: **Module 5, campaign incrementality**. No promotion causal effect, profit estimate, or ROI has been established in this module.